In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json
import networkx as nx
import numpy as np
import ast
import pickle

from src.functions import *

In [ ]:
REPORT_DATA_FILEPATH = "data/reports.csv"
FIX_DATA_FILEPATH = "data/fixes.csv"
TASK_DATA_FILEPATH = "data/tasks.csv"

## Data Cleaning

In [ ]:
report_data = pd.read_csv(REPORT_DATA_FILEPATH, sep=";")
fix_data = pd.read_csv(FIX_DATA_FILEPATH, sep=";")
task_data = pd.read_csv(TASK_DATA_FILEPATH, sep=";")


def parse_list_string(val):
    if pd.isna(val):
        return []
    return ast.literal_eval(val)


report_data["changelog"] = report_data["changelog"].apply(parse_list_string)
report_data["linked_fixes"] = report_data["linked_fixes"].apply(parse_list_string)
report_data["linked_tasks"] = report_data["linked_tasks"].apply(parse_list_string)
fix_data["changelog"] = fix_data["changelog"].apply(parse_list_string)
task_data["labels"] = task_data["labels"].apply(parse_list_string)

In [ ]:
task_data["shared_reports"] = 0
fix_data["shared_reports"] = 0

for _, row in report_data.iterrows():
    for task in row["linked_tasks"]:
        task_data.loc[task_data["key"] == task, "shared_reports"] += 1
    for fix in row["linked_fixes"]:
        fix_data.loc[fix_data["key"] == fix, "shared_reports"] += 1

In [ ]:
report_data["task_timespent"] = report_data["key"].apply(
    lambda key: get_report_timespent_from_task(key, report_data, task_data)
)

report_data["defect_path"] = report_data["key"].apply(
    lambda key: get_path(key, report_data)
)

report_data["defect_age"] = report_data["defect_path"].apply(
    lambda x: len(x) - 1 if isinstance(x, list) else None
)

report_data.head()

In [ ]:
excluded_should_have_been_found = [
    "Not a bug / Problem not reproducible",
    "Not Relevant: 3rd party SW/HW",
]

excluded_detection_phases = [
    "HW: Integration / System testing",
]

excluded_defect_causes = [
    "Misunderstanding by end user",
    "No Root Cause",
]

### Get machine-learnable data

In [ ]:
severity_map = {
    "LOW": 0,
    "SEVERE": 1,
    "FATAL": 2,
}

priority_map = {
    "Minor": 0,
    "Normal": 1,
    "Critical": 2,
    "Emergency": 3,
}

detection_phase_map = {
    "Implementation": 0,
    "Integration": 1,
    "Verification": 2,
    "Field validation": 3,
    "Customer": 4,
}

shbf_map = {
    "SFS / SFIS review": 0,
    "AS review": 0,
    "ID review": 0,
    "Code review": 0,
    "SW Unit / Module testing": 0,
    "SW I&V / Smoke testing": 0,
    "I&V feature / integration testing": 1,
    "I&V System testing": 2,
    "I&V Robustness / Stability testing": 2,
    "I&V Capacity / Performance testing": 2,
    "I&V Regression / Upgrade testing": 2,
    "CD testing (maintenance)": 2,
    "CuDo reviews / testing": 2,
    "IoP testing": 3,
    "Field Validation / pilot testing": 3,
    "E2E validation": 3,
}

In [ ]:
ml_df = report_data[["num_comments", "num_commenters"]]
ml_df["severity"] = report_data["severity"].map(lambda x: severity_map[x])
ml_df["priority"] = report_data["priority"].map(lambda x: priority_map[x])
ml_df["should_have_been_found"] = report_data["should_have_been_found"].map(
    lambda x: shbf_map[x] if x not in excluded_should_have_been_found else None
)
ml_df["detection_phase"] = report_data["detection_phase"].map(
    lambda x: detection_phase_map[x] if x not in excluded_detection_phases else None
)
ml_df["reached_customer"] = report_data["detection_phase"].map(
    lambda x: 1 if x == "Customer" else 0
)
ml_df["len_changelog"] = report_data["changelog"].map(lambda x: len(x))
ml_df["num_fixes"] = report_data["linked_fixes"].map(lambda x: len(x))
ml_df["num_tasks"] = report_data["linked_tasks"].map(lambda x: len(x))
ml_df["task_timespent"] = report_data["task_timespent"]
ml_df = ml_df.sort_values(by=["task_timespent"])
ml_df = ml_df.reset_index(drop=True)
ml_df.to_csv("gen/ml_timespent.csv", index=False)

In [ ]:
# Get json file that maps Should Have Been Found, Detection phase, and
# Defect Root Cause data to expected defect path
with open("src/path_mapping.json", "r", encoding="utf-8") as f:
    mapping = json.load(f)

### Export viable ML data

In [ ]:
defect_ml_data = report_data[["defect_path", "task_timespent"]]
defect_ml_data = defect_ml_data.dropna()

defect_ml_data.to_csv("gen/ml_data.csv", index=False)

In [ ]:
print(f"Number of viable paths: {len(report_data.dropna(subset=['defect_path']))}")
print(f"Number of complete samples: {len(defect_ml_data)}")

### Create network

In [ ]:
G = nx.DiGraph()
starting_weight = 1

# Generate directed graph based on phases txt
with open("src/phases.txt", "r") as f:
    lines = [line.strip() for line in f.readlines() if line.strip()]

G.add_node("SPAWN", layer=0)
G.add_node("DETECTION", layer=len(lines) + 1)

prev_nodes = []
for i, line in enumerate(lines):
    strings = line.split(" ")
    phase = strings[0]
    teams = strings[1:]

    current_nodes = []
    for team in teams:
        node_name = phase + "_" + team
        G.add_node(node_name, layer=i + 1)
        current_nodes.append(node_name)

    # Add phase-to-phase connections
    if prev_nodes:
        for u in prev_nodes:
            for v in current_nodes:
                G.add_edge(u, v, weight=starting_weight)

    # Add SPAWN connections
    is_last_line = i == len(lines) - 1
    if not is_last_line:
        for v in current_nodes:
            G.add_edge("SPAWN", v, weight=starting_weight)

    # Add DETECTION connections
    is_first_line = i == 0
    if not is_first_line:
        for u in current_nodes:
            G.add_edge(u, "DETECTION", weight=starting_weight)

    prev_nodes = current_nodes

# Add to edge weights based on report data
for i, report in report_data.iterrows():
    start = report["should_have_been_found"]
    end = report["detection_phase"]
    cause = report["defect_root_cause"]

    if start in mapping.keys() and end in mapping[start].keys():
        # Check if there's a specific mapping for defect root cause
        if (
            isinstance(mapping[start][end], dict)
            and cause in mapping[start][end].keys()
        ):
            path_template = mapping[start][end][cause]
        # If not, use only start and end
        elif isinstance(mapping[start][end], list):
            path_template = mapping[start][end]
        else:
            continue

        # Get explicit defect path by unpacking it
        path = unpack_path(path_template)

        # Add a weight to each path
        for i in range(len(path) - 1):
            source_strings = path[i]
            target_strings = path[i + 1]

            source_nodes = [
                node
                for node in G.nodes()
                if any(node == s or node.startswith(f"{s}_") for s in source_strings)
            ]

            target_nodes = [
                node
                for node in G.nodes()
                if any(node == s or node.startswith(f"{s}_") for s in target_strings)
            ]

            # Distribute 1 weight across all connections
            num_connections = len(source_nodes) * len(target_nodes)
            for u in source_nodes:
                for v in target_nodes:
                    G[u][v]["weight"] += 1 / num_connections

# Normalize leading-out edges
for node in G.nodes():
    edges_out = G.out_edges(node)
    neighbors = [neighbor for (_, neighbor) in edges_out]
    total_weight = np.sum([G[node][neighbor]["weight"] for neighbor in neighbors])
    if total_weight != 0:
        for neighbor in neighbors:
            G[node][neighbor]["prob"] = G[node][neighbor]["weight"] / total_weight

In [ ]:
# Export graph
pickle.dump(G, open("gen/network.pickle", "wb"))

In [ ]:
pr = nx.pagerank(G, alpha=0.8)
ranks = [(node, pr[node]) for node in G.nodes()]
print("Defect PageRanks:")
for node, rank in sorted(ranks, key=lambda x: x[1], reverse=True):
    print(f"{node}: {rank:.4f}")

### Render graphs for thesis

In [ ]:
TITLE_PARAMS = {"fontsize": "14", "fontweight": "semibold"}
LABEL_PARAMS = {"fontsize": "12"}

cm = plt.cm.viridis(np.linspace(0, 1, 5))

In [ ]:
plt.figure(figsize=(10, 5))
defect_cost_data = report_data[["defect_age", "task_timespent"]].dropna()
defect_cost_data["defect_age"] = defect_cost_data["defect_age"].astype(int)

sns.violinplot(data=defect_cost_data, x="defect_age", y="task_timespent", color=cm[2])

plt.ylim(0, 130)
plt.title("Defect Cost by Age", **TITLE_PARAMS)
plt.xlabel("Defect Age", **LABEL_PARAMS)
plt.ylabel("Defect Cost (h)", **LABEL_PARAMS)
plt.tight_layout()

plt.savefig("fig_defect_costs_by_age.png")
plt.show()

In [ ]:
path_stats = report_data[["defect_path", "task_timespent"]].dropna(
    subset=["defect_path"]
)
path_stats["age"] = path_stats["defect_path"].map(lambda x: len(x) - 1)
age_nan = path_stats[path_stats["task_timespent"].isna()]["age"]
age_not_nan = path_stats[path_stats["task_timespent"].notna()]["age"]

plt.figure(figsize=(10, 6))
plt.hist(
    [age_nan, age_not_nan],
    stacked=True,
    color=[cm[0], cm[2]],
    bins=[0.5, 1.5, 2.5, 3.5, 4.5, 5.5],
    rwidth=0.8,
    label=["Defects without logged costs", "Defects with logged costs"],
)

plt.xticks([1, 2, 3, 4, 5])
plt.title("Defect Age Distribution", **TITLE_PARAMS)
plt.xlabel("Defect Age", **LABEL_PARAMS)
plt.ylabel("Number of Defects", **LABEL_PARAMS)
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig("fig_defect_age_distribution.png")
plt.show()

In [ ]:
pressman_phases = ["Design", "Implementation", "Testing", "Maintenance"]
pressman_vals = [1, 6.5, 15, 100]
plt.figure(figsize=(10, 5))
bars = plt.bar(
    pressman_phases,
    pressman_vals,
    color=np.flip(plt.cm.viridis(np.linspace(0, 1, 6))[:4], 0),
)
plt.bar_label(bars, fontsize=11)
plt.title("Relative Cost of Fixing Defects", **TITLE_PARAMS)
plt.tight_layout()
plt.savefig("fig_pressman.png")
plt.show()